# Keyword Price Outliers With Name Blacklist Filter

`keyword_price_outliers.ipynb` 흐름을 복사하되, 가격 이상치 분석 전에 크롤링 단계에서 제거할 이름 블랙리스트 필터를 먼저 적용합니다.

블랙리스트 필터 기준:
- `DROP_NAME_KEYWORDS`에 제외할 단어를 입력합니다.
- 크롤링된 `name`에 해당 단어가 포함되면 드랍합니다.
- 예: 케이스, 필름, 강화유리, 맥세이프, 충전기 등 본체 가격 통계를 흐리는 액세서리/구매글/홍보 문구

이 노트북은 CSV 저장보다 결과 확인용 DataFrame 출력에 초점을 둡니다.


In [1]:
from pathlib import Path
import re

import pandas as pd


def resolve_analysis_dir() -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name == "analysis":
        return cwd

    candidate = cwd / "code/backend/src/main/python/analysis"
    if candidate.exists():
        return candidate

    return Path(__file__).resolve().parent if "__file__" in globals() else cwd


ANALYSIS_DIR = resolve_analysis_dir()
PYTHON_DIR = ANALYSIS_DIR.parent
CRAWLING_RESULTS_DIR = PYTHON_DIR / "crawling" / "results"
TARGET_CSV_NAME = "통합조회_전체_no_filter_20260605_1142.csv"

csv_path = CRAWLING_RESULTS_DIR / TARGET_CSV_NAME
if not csv_path.exists():
    raise FileNotFoundError(f"지정한 no-filter 크롤링 CSV를 찾을 수 없습니다: {csv_path}")

csv_path


WindowsPath('C:/project/kdtproject/kdtproject/code/backend/src/main/python/crawling/results/통합조회_전체_no_filter_20260601_1815.csv')

In [2]:
# 여기에 크롤링 단계에서 제외하고 싶은 name 포함 키워드를 추가하세요.
DROP_NAME_KEYWORDS = [
    "삽니다",
    "구매합니다",
    "매입",
    "대여",
    "교환",
    "렌탈",
    "매입",
    "대여",
]


def normalize_search_text(value):
    text = str(value or "").lower()
    text = re.sub(r"(?<=[a-z0-9])plus\b", " 플러스", text)
    text = re.sub(r"\bplus\b|[+＋]", " 플러스 ", text)
    text = re.sub(r"\bpro\b", " 프로 ", text)
    text = re.sub(r"\bmax\b", " 맥스 ", text)
    text = re.sub(r"\bultra\b", " 울트라 ", text)
    text = re.sub(r"[^0-9a-z가-힣\s]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def compact_search_text(value):
    return re.sub(r"\s+", "", normalize_search_text(value))


def build_drop_name_keywords(keywords):
    drop_keywords = []
    seen = set()

    for keyword in keywords:
        original = str(keyword or "").strip()
        normalized = normalize_search_text(original)
        compact = compact_search_text(original)
        key = (normalized, compact)

        if not normalized or key in seen:
            continue

        seen.add(key)
        drop_keywords.append(
            {
                "original": original,
                "normalized": normalized,
                "compact": compact,
            }
        )

    return drop_keywords


DROP_NAME_KEYWORD_MATCHERS = build_drop_name_keywords(DROP_NAME_KEYWORDS)
NORMALIZED_DROP_NAME_KEYWORDS = [keyword["normalized"] for keyword in DROP_NAME_KEYWORD_MATCHERS]


def matched_drop_keywords(name):
    raw_name = str(name or "").lower()
    normalized_name = normalize_search_text(name)
    compact_name = compact_search_text(name)

    matched_keywords = []
    for keyword in DROP_NAME_KEYWORD_MATCHERS:
        raw_keyword = keyword["original"].lower()
        normalized_keyword = keyword["normalized"]
        compact_keyword = keyword["compact"]

        if (
            raw_keyword in raw_name
            or normalized_keyword in normalized_name
            or compact_keyword in compact_name
        ):
            matched_keywords.append(keyword["original"])

    return matched_keywords


In [3]:
df = pd.read_csv(csv_path, encoding="utf-8-sig")

working_df = df.copy()
working_df["price_numeric"] = pd.to_numeric(working_df["price"], errors="coerce")
working_df = working_df.dropna(subset=["keyword", "name", "price_numeric"])
working_df = working_df[working_df["price_numeric"] > 0].copy()
working_df["price_numeric"] = working_df["price_numeric"].astype(int)

working_df["matched_drop_keywords"] = working_df["name"].apply(matched_drop_keywords)
working_df["name_blacklist_drop"] = working_df["matched_drop_keywords"].apply(bool)
working_df["matched_drop_keywords_text"] = working_df["matched_drop_keywords"].apply(lambda values: " | ".join(values))

filtered_df = working_df[~working_df["name_blacklist_drop"]].copy()
dropped_filter_df = working_df[working_df["name_blacklist_drop"]].copy()

filter_overview_df = pd.DataFrame([
    {
        "source_file": csv_path.name,
        "valid_price_rows_before_filter": len(working_df),
        "rows_after_name_blacklist_filter": len(filtered_df),
        "removed_by_name_blacklist_filter": len(dropped_filter_df),
        "name_blacklist_keep_rate": len(filtered_df) / len(working_df) if len(working_df) else 0,
        "keyword_count_before_filter": working_df["keyword"].nunique(),
        "keyword_count_after_filter": filtered_df["keyword"].nunique(),
        "drop_keyword_count": len(NORMALIZED_DROP_NAME_KEYWORDS),
    }
])

filter_overview_df


,source_file,valid_price_rows_before_filter,rows_after_name_blacklist_filter,removed_by_name_blacklist_filter,name_blacklist_keep_rate,keyword_count_before_filter,keyword_count_after_filter,drop_keyword_count
0,통합조회_전체_no_filter_20260601_1815.csv,24906,24233,673,0.972978,20,20,6


In [4]:
filter_keyword_summary_df = working_df.groupby("keyword").agg(
    before_count=("keyword", "count"),
    dropped_by_name_blacklist=("name_blacklist_drop", "sum"),
).reset_index()
filter_keyword_summary_df["after_name_blacklist_count"] = (
    filter_keyword_summary_df["before_count"] - filter_keyword_summary_df["dropped_by_name_blacklist"]
)
filter_keyword_summary_df["name_blacklist_drop_rate"] = (
    filter_keyword_summary_df["dropped_by_name_blacklist"] / filter_keyword_summary_df["before_count"]
)
filter_keyword_summary_df = filter_keyword_summary_df.sort_values(
    ["dropped_by_name_blacklist", "before_count"], ascending=[False, False]
)

filter_keyword_summary_df


,keyword,before_count,dropped_by_name_blacklist,after_name_blacklist_count,name_blacklist_drop_rate
16,스텔라이브,1869,293,1576,0.156768
4,갤럭시,10464,137,10327,0.013093
0,5090,264,46,218,0.174242
6,골드바,542,36,506,0.066421
14,세이코,2030,35,1995,0.017241
5,게이밍 노트북,957,34,923,0.035528
2,ps5,1040,16,1024,0.015385
19,핫토이,1726,15,1711,0.008691
7,던스,1228,11,1217,0.008958
15,센티넬,184,9,175,0.048913


In [5]:
# 1차 name 블랙리스트 필터 적용 후 keyword별 저가 데이터 조회
LOW_PRICE_PER_KEYWORD_LIMIT = 30
LOW_PRICE_KEYWORD = None  # 예: "갤럭시", "5090". 전체 keyword 조회는 None

low_price_review_base_df = filtered_df.copy()

if LOW_PRICE_KEYWORD:
    low_price_review_base_df = low_price_review_base_df[
        low_price_review_base_df["keyword"].astype(str).str.contains(LOW_PRICE_KEYWORD, case=False, na=False)
    ]

low_price_review_df = (
    low_price_review_base_df.sort_values(["keyword", "price_numeric", "platform", "pid"])
    .groupby("keyword", group_keys=False)
    .head(LOW_PRICE_PER_KEYWORD_LIMIT)
    .copy()
)
low_price_review_df["low_price_rank_in_keyword"] = (
    low_price_review_df.groupby("keyword").cumcount() + 1
)

low_price_review_df = low_price_review_df[
    [
        "keyword",
        "low_price_rank_in_keyword",
        "platform",
        "pid",
        "name",
        "price_numeric",
        "status",
        "link",
    ]
].sort_values(["keyword", "low_price_rank_in_keyword"])

print(
    f"조회 keyword 수: {low_price_review_df['keyword'].nunique():,}, "
    f"조회 row 수: {len(low_price_review_df):,}"
)

with pd.option_context("display.max_rows", len(low_price_review_df), "display.max_colwidth", 120):
    display(low_price_review_df)


조회 keyword 수: 20, 조회 row 수: 600


,keyword,low_price_rank_in_keyword,platform,pid,name,price_numeric,status,link
5791,5090,1,중고나라,223733140,사기 5070 5080 4090 5090 9950 9800 p41 p51 990 9910,112,판매중,https://web.joongna.com/product/223733140
5744,5090,2,중고나라,229040751,이엠텍 팰릿 RTX 5090 게임락 32GB 그래픽카드,1004,거래완료,https://web.joongna.com/product/229040751
5607,5090,3,번개장터,407882991,"9950x3d, x870e, 64gb, 2tb, 5090 하이엔드 본체",1111,예약중,https://m.bunjang.co.kr/products/407882991
5651,5090,4,번개장터,408316423,"9800x3d, B850m wifi, rtx 5080 화이트 완본체",1111,예약중,https://m.bunjang.co.kr/products/408316423
5632,5090,5,번개장터,409072691,"9800x3d, b850m wifi, rtx 5070ti 고사양 본체",1111,예약중,https://m.bunjang.co.kr/products/409072691
5601,5090,6,번개장터,410038131,"9800x3d, b850m wifi, rtx 5070ti 갤럭시 본체",1111,예약중,https://m.bunjang.co.kr/products/410038131
5581,5090,7,번개장터,410875935,"9800x3d, b850m wifi, rtx 5080 초고사양 본체",1111,예약중,https://m.bunjang.co.kr/products/410875935
5570,5090,8,번개장터,410876033,"9800x3d, b850m wifi, rtx 5080 화이트 초고사양pc",1111,예약중,https://m.bunjang.co.kr/products/410876033
5571,5090,9,번개장터,410876316,"9950x3d, x870e wifi, 2tb, 5080 하이엔드급 본체",1111,예약중,https://m.bunjang.co.kr/products/410876316
5743,5090,10,중고나라,228924078,"9950x3d, x870e, 64gb, 2tb, 5090 하이엔드 본체",1111,거래완료,https://web.joongna.com/product/228924078


In [6]:
dropped_filter_df[[
    "keyword",
    "platform",
    "pid",
    "name",
    "price_numeric",
    "matched_drop_keywords_text",
    "status",
    "link",
]].sort_values(["keyword", "platform", "price_numeric"]).head(100)


,keyword,platform,pid,name,price_numeric,matched_drop_keywords_text,status,link
5615,5090,번개장터,390058761,(가격인하) Rtx5080 만기인수형 장기렌탈,132900,렌탈,판매중,https://m.bunjang.co.kr/products/390058761
5614,5090,번개장터,389838065,Rtx5090 만기인수형 장기렌탈,360000,렌탈,판매중,https://m.bunjang.co.kr/products/389838065
5629,5090,번개장터,405956469,"그래픽카드 삽니다. (5070ti, 5080, 5090)",2000000,삽니다,판매중,https://m.bunjang.co.kr/products/405956469
5548,5090,번개장터,411671068,RTX 5080 어로스화이트+350 5090어로스 화이트삽니다,3500000,삽니다,판매중,https://m.bunjang.co.kr/products/411671068
5598,5090,번개장터,362210598,삽니다 5090 구매원합니다,5000000,삽니다,판매중,https://m.bunjang.co.kr/products/362210598
...,...,...,...,...,...,...,...,...
18926,갤럭시,번개장터,388883327,보증금x) 갤럭시 S22 울트라 대여,15000,대여,판매중,https://m.bunjang.co.kr/products/388883327
16795,갤럭시,번개장터,396405633,갤럭시 S22 울트라 대여,16000,대여,판매중,https://m.bunjang.co.kr/products/396405633
13774,갤럭시,번개장터,380415826,갤럭시 s25 울트라 대여 해드립니다 위버스콘 재현 알디원 워너원고,19000,대여,판매중,https://m.bunjang.co.kr/products/380415826
14630,갤럭시,번개장터,281560620,[대여] 갤럭시S23 울트라,19000,대여,판매중,https://m.bunjang.co.kr/products/281560620


In [7]:
LOW_PRICE_MEDIAN_RATIO = 0.2

keyword_stats_df = (
    filtered_df.groupby("keyword")["price_numeric"]
    .agg(
        item_count="count",
        min_price="min",
        q1=lambda values: values.quantile(0.25),
        median_price="median",
        q3=lambda values: values.quantile(0.75),
        max_price="max",
        average_price="mean",
    )
    .reset_index()
)

keyword_stats_df["iqr"] = keyword_stats_df["q3"] - keyword_stats_df["q1"]
keyword_stats_df["iqr_lower_bound"] = (keyword_stats_df["q1"] - 1.5 * keyword_stats_df["iqr"]).clip(lower=0)
keyword_stats_df["median_ratio_lower_bound"] = keyword_stats_df["median_price"] * LOW_PRICE_MEDIAN_RATIO
keyword_stats_df["lower_bound"] = keyword_stats_df[["iqr_lower_bound", "median_ratio_lower_bound"]].max(axis=1)
keyword_stats_df["upper_bound"] = keyword_stats_df["q3"] + 1.5 * keyword_stats_df["iqr"]

keyword_stats_df.sort_values(["item_count", "keyword"], ascending=[False, True])


,keyword,item_count,min_price,q1,median_price,q3,max_price,average_price,iqr,iqr_lower_bound,median_ratio_lower_bound,lower_bound,upper_bound
4,갤럭시,10327,500,90000.0,160000.0,338900.0,999999999,3.738847e+05,248900.0,0.0,32000.0,32000.0,712250.0
14,세이코,1995,111,130000.0,240000.0,500000.0,999999999,1.106652e+06,370000.0,0.0,48000.0,48000.0,1055000.0
19,핫토이,1711,100,114000.0,225000.0,365000.0,999999999,9.036770e+05,251000.0,0.0,45000.0,45000.0,741500.0
16,스텔라이브,1576,500,13000.0,30000.0,70000.0,999999999,2.358331e+06,57000.0,0.0,6000.0,6000.0,155500.0
7,던스,1217,1,30000.0,59000.0,100000.0,2500000,8.775643e+04,70000.0,0.0,11800.0,11800.0,205000.0
8,만년필,1111,1,45000.0,150000.0,500000.0,99999999,8.144601e+05,455000.0,0.0,30000.0,30000.0,1182500.0
2,ps5,1024,2222,23000.0,40000.0,90000.0,6850000,1.501343e+05,67000.0,0.0,8000.0,8000.0,190500.0
5,게이밍 노트북,923,1,450000.0,880000.0,1475000.0,9500000,1.111335e+06,1025000.0,0.0,176000.0,176000.0,3012500.0
12,바이씨니,890,1,50000.0,70000.0,104250.0,999999,8.652195e+04,54250.0,0.0,14000.0,14000.0,185625.0
18,제라늄,781,1111,8000.0,12000.0,24000.0,1234567,2.982699e+04,16000.0,0.0,2400.0,2400.0,48000.0


In [8]:
outlier_df = filtered_df.merge(
    keyword_stats_df[[
        "keyword",
        "q1",
        "median_price",
        "q3",
        "iqr",
        "iqr_lower_bound",
        "median_ratio_lower_bound",
        "lower_bound",
        "upper_bound",
    ]],
    on="keyword",
    how="left",
)

outlier_df["outlier_type"] = ""
outlier_df.loc[outlier_df["price_numeric"] < outlier_df["lower_bound"], "outlier_type"] = "low"
outlier_df.loc[outlier_df["price_numeric"] > outlier_df["upper_bound"], "outlier_type"] = "high"
outlier_df = outlier_df[outlier_df["outlier_type"] != ""].copy()
outlier_df["price_to_median_ratio"] = outlier_df["price_numeric"] / outlier_df["median_price"]

outlier_df[[
    "keyword",
    "platform",
    "pid",
    "name",
    "price_numeric",
    "outlier_type",
    "lower_bound",
    "median_price",
    "upper_bound",
    "price_to_median_ratio",
    "status",
    "link",
]].sort_values(["keyword", "outlier_type", "price_numeric"]).head(100)


,keyword,platform,pid,name,price_numeric,outlier_type,lower_bound,median_price,upper_bound,price_to_median_ratio,status,link
5239,5090,번개장터,409938598,"ASUS ROG 9950X3D, 5090 아스트랄 BTF",13297870,high,1285000.0,6425000.0,13253125.0,2.069707,판매중,https://m.bunjang.co.kr/products/409938598
5375,5090,중고나라,223733140,사기 5070 5080 4090 5090 9950 9800 p41 p51 990 9910,112,low,1285000.0,6425000.0,13253125.0,0.000017,판매중,https://web.joongna.com/product/223733140
5338,5090,중고나라,229040751,이엠텍 팰릿 RTX 5090 게임락 32GB 그래픽카드,1004,low,1285000.0,6425000.0,13253125.0,0.000156,거래완료,https://web.joongna.com/product/229040751
5193,5090,번개장터,410876033,"9800x3d, b850m wifi, rtx 5080 화이트 초고사양pc",1111,low,1285000.0,6425000.0,13253125.0,0.000173,예약중,https://m.bunjang.co.kr/products/410876033
5194,5090,번개장터,410876316,"9950x3d, x870e wifi, 2tb, 5080 하이엔드급 본체",1111,low,1285000.0,6425000.0,13253125.0,0.000173,예약중,https://m.bunjang.co.kr/products/410876316
...,...,...,...,...,...,...,...,...,...,...,...,...
5022,a7c2,번개장터,400375253,@ 현대 아반떼 CN7 운전석 13핀 LED라이트 92101-AA700,176000,low,360000.0,1800000.0,4870000.0,0.097778,판매중,https://m.bunjang.co.kr/products/400375253
5053,a7c2,번개장터,392644109,아우디 A7 C8 4K 2세대 그릴 4K8 853 651 A 중고부품,180000,low,360000.0,1800000.0,4870000.0,0.100000,판매중,https://m.bunjang.co.kr/products/392644109
4989,a7c2,번개장터,397384176,캐논 파워샷 Canon Powershot a720is 카메라 판매합니다,191480,low,360000.0,1800000.0,4870000.0,0.106378,판매중,https://m.bunjang.co.kr/products/397384176
5041,a7c2,번개장터,410289470,아우디 a7 4g8 c7 전반기2011-2014 앞범퍼 전피 밤바,200000,low,360000.0,1800000.0,4870000.0,0.111111,판매중,https://m.bunjang.co.kr/products/410289470


In [9]:
# 가격 이상치 name 분석용 조회 셀
# 필요하면 아래 조건만 바꿔서 다시 실행하세요.
OUTLIER_KEYWORD = None  # 예: "갤럭시", "5090". 전체 조회는 None
OUTLIER_TYPE = None  # "low", "high" 또는 전체 조회는 None
NAME_CONTAINS = ""  # name에 포함된 단어로 추가 검색. 전체 조회는 빈 문자열
OUTLIER_LIMIT = 300

price_outlier_review_df = outlier_df.copy()
price_outlier_review_df["price_gap_from_median"] = (
    price_outlier_review_df["price_numeric"] - price_outlier_review_df["median_price"]
)
price_outlier_review_df["outlier_severity"] = price_outlier_review_df.apply(
    lambda row: row["median_price"] / row["price_numeric"]
    if row["outlier_type"] == "low"
    else row["price_to_median_ratio"],
    axis=1,
)

if OUTLIER_KEYWORD:
    price_outlier_review_df = price_outlier_review_df[
        price_outlier_review_df["keyword"].astype(str).str.contains(OUTLIER_KEYWORD, case=False, na=False)
    ]

if OUTLIER_TYPE:
    price_outlier_review_df = price_outlier_review_df[
        price_outlier_review_df["outlier_type"].eq(OUTLIER_TYPE)
    ]

if NAME_CONTAINS:
    price_outlier_review_df = price_outlier_review_df[
        price_outlier_review_df["name"].astype(str).str.contains(NAME_CONTAINS, case=False, na=False)
    ]

price_outlier_review_df = price_outlier_review_df[
    [
        "keyword",
        "outlier_type",
        "platform",
        "pid",
        "name",
        "price_numeric",
        "median_price",
        "price_gap_from_median",
        "price_to_median_ratio",
        "outlier_severity",
        "lower_bound",
        "upper_bound",
        "status",
        "link",
    ]
].sort_values(
    ["keyword", "outlier_type", "outlier_severity"],
    ascending=[True, True, False],
)

print(f"조회된 가격 이상치 수: {len(price_outlier_review_df):,}")
price_outlier_review_df.head(OUTLIER_LIMIT)


조회된 가격 이상치 수: 4,847


,keyword,outlier_type,platform,pid,name,price_numeric,median_price,price_gap_from_median,price_to_median_ratio,outlier_severity,lower_bound,upper_bound,status,link
5239,5090,high,번개장터,409938598,"ASUS ROG 9950X3D, 5090 아스트랄 BTF",13297870,6425000.0,6872870.0,2.069707,2.069707,1285000.0,13253125.0,판매중,https://m.bunjang.co.kr/products/409938598
5375,5090,low,중고나라,223733140,사기 5070 5080 4090 5090 9950 9800 p41 p51 990 9910,112,6425000.0,-6424888.0,0.000017,57366.071429,1285000.0,13253125.0,판매중,https://web.joongna.com/product/223733140
5338,5090,low,중고나라,229040751,이엠텍 팰릿 RTX 5090 게임락 32GB 그래픽카드,1004,6425000.0,-6423996.0,0.000156,6399.402390,1285000.0,13253125.0,거래완료,https://web.joongna.com/product/229040751
5193,5090,low,번개장터,410876033,"9800x3d, b850m wifi, rtx 5080 화이트 초고사양pc",1111,6425000.0,-6423889.0,0.000173,5783.078308,1285000.0,13253125.0,예약중,https://m.bunjang.co.kr/products/410876033
5194,5090,low,번개장터,410876316,"9950x3d, x870e wifi, 2tb, 5080 하이엔드급 본체",1111,6425000.0,-6423889.0,0.000173,5783.078308,1285000.0,13253125.0,예약중,https://m.bunjang.co.kr/products/410876316
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6403,ps5,low,번개장터,265047601,PS5 커버 Cover,7000,40000.0,-33000.0,0.175000,5.714286,8000.0,190500.0,판매중,https://m.bunjang.co.kr/products/265047601
6459,ps5,low,번개장터,399387790,PS5 NBA 2K23 농구 게임 한글지원,7000,40000.0,-33000.0,0.175000,5.714286,8000.0,190500.0,판매중,https://m.bunjang.co.kr/products/399387790
4835,y700,high,번개장터,410370373,267001975 디올 레이디백 체인 미니 2way 토트백 그레이,2700000,350000.0,2350000.0,7.714286,7.714286,70000.0,1010000.0,판매중,https://m.bunjang.co.kr/products/410370373
4859,y700,high,번개장터,406579849,267001716 디올 마이크로 레이디 디올 램스킨 핸드백 2way,2400000,350000.0,2050000.0,6.857143,6.857143,70000.0,1010000.0,판매중,https://m.bunjang.co.kr/products/406579849


In [10]:
outlier_summary_df = (
    outlier_df.groupby(["keyword", "outlier_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for column in ["low", "high"]:
    if column not in outlier_summary_df.columns:
        outlier_summary_df[column] = 0

outlier_summary_df["total_outlier_count"] = outlier_summary_df["low"] + outlier_summary_df["high"]
outlier_summary_df = outlier_summary_df.merge(
    keyword_stats_df[["keyword", "item_count", "min_price", "median_price", "max_price", "lower_bound", "upper_bound"]],
    on="keyword",
    how="left",
)
outlier_summary_df["outlier_rate"] = outlier_summary_df["total_outlier_count"] / outlier_summary_df["item_count"]

outlier_overview_df = outlier_summary_df[[
    "keyword",
    "item_count",
    "low",
    "high",
    "total_outlier_count",
    "outlier_rate",
    "min_price",
    "median_price",
    "max_price",
    "lower_bound",
    "upper_bound",
]].sort_values(["total_outlier_count", "outlier_rate"], ascending=[False, False])

overall_outlier_overview_df = pd.DataFrame([
    {
        "source_file": csv_path.name,
        "valid_price_rows_before_filter": len(working_df),
        "rows_after_name_blacklist_filter": len(filtered_df),
        "removed_by_name_blacklist_filter": len(dropped_filter_df),
        "name_blacklist_keep_rate": len(filtered_df) / len(working_df) if len(working_df) else 0,
        "outlier_rows_after_name_blacklist_filter": len(outlier_df),
        "outlier_rate_after_name_blacklist_filter": len(outlier_df) / len(filtered_df) if len(filtered_df) else 0,
        "low_outlier_rows": int((outlier_df["outlier_type"] == "low").sum()) if len(outlier_df) else 0,
        "high_outlier_rows": int((outlier_df["outlier_type"] == "high").sum()) if len(outlier_df) else 0,
        "low_price_median_ratio": LOW_PRICE_MEDIAN_RATIO,
    }
])

display(overall_outlier_overview_df)
outlier_overview_df


,source_file,valid_price_rows_before_filter,rows_after_name_blacklist_filter,removed_by_name_blacklist_filter,name_blacklist_keep_rate,outlier_rows_after_name_blacklist_filter,outlier_rate_after_name_blacklist_filter,low_outlier_rows,high_outlier_rows,low_price_median_ratio
0,통합조회_전체_no_filter_20260601_1815.csv,24906,24233,673,0.972978,4847,0.200017,2712,2135,0.2


,keyword,item_count,low,high,total_outlier_count,outlier_rate,min_price,median_price,max_price,lower_bound,upper_bound
4,갤럭시,10327,1317,891,2208,0.213808,500,160000.0,999999999,32000.0,712250.0
16,스텔라이브,1576,220,176,396,0.251269,500,30000.0,999999999,6000.0,155500.0
14,세이코,1995,152,233,385,0.192982,111,240000.0,999999999,48000.0,1055000.0
19,핫토이,1711,259,116,375,0.219170,100,225000.0,999999999,45000.0,741500.0
8,만년필,1111,182,118,300,0.270027,1,150000.0,99999999,30000.0,1182500.0
2,ps5,1024,18,172,190,0.185547,2222,40000.0,6850000,8000.0,190500.0
6,골드바,506,106,79,185,0.365613,250,815000.0,999999999,163000.0,1862500.0
7,던스,1217,71,73,144,0.118324,1,59000.0,2500000,11800.0,205000.0
5,게이밍 노트북,923,81,41,122,0.132178,1,880000.0,9500000,176000.0,3012500.0
18,제라늄,781,13,92,105,0.134443,1111,12000.0,1234567,2400.0,48000.0


In [11]:
# name 블랙리스트 필터 + 가격 이상치 제거 후 keyword별 가격 요약 DF
outlier_keys = outlier_df[["keyword", "platform", "pid", "price_numeric"]].copy()
outlier_keys["is_outlier"] = True

clean_price_df = filtered_df.merge(
    outlier_keys[["keyword", "platform", "pid", "price_numeric", "is_outlier"]],
    on=["keyword", "platform", "pid", "price_numeric"],
    how="left",
)
clean_price_df["is_outlier"] = clean_price_df["is_outlier"].fillna(False).astype(bool)
clean_price_df = clean_price_df.loc[~clean_price_df["is_outlier"]].copy()

clean_keyword_price_summary_df = (
    clean_price_df.groupby("keyword")["price_numeric"]
    .agg(
        clean_item_count="count",
        clean_min_price="min",
        clean_max_price="max",
        clean_average_price="mean",
        clean_median_price="median",
    )
    .reset_index()
)

filtered_counts_df = filtered_df.groupby("keyword").size().reset_index(name="name_blacklist_filtered_item_count")
outlier_counts_df = outlier_df.groupby("keyword").size().reset_index(name="removed_price_outlier_count")

clean_keyword_price_summary_df = filtered_counts_df.merge(
    outlier_counts_df,
    on="keyword",
    how="left",
).merge(
    clean_keyword_price_summary_df,
    on="keyword",
    how="left",
)
clean_keyword_price_summary_df["removed_price_outlier_count"] = clean_keyword_price_summary_df["removed_price_outlier_count"].fillna(0).astype(int)
clean_keyword_price_summary_df["clean_item_count"] = clean_keyword_price_summary_df["clean_item_count"].fillna(0).astype(int)
clean_keyword_price_summary_df["removed_price_outlier_rate"] = clean_keyword_price_summary_df["removed_price_outlier_count"] / clean_keyword_price_summary_df["name_blacklist_filtered_item_count"]

clean_keyword_price_summary_df = clean_keyword_price_summary_df[[
    "keyword",
    "name_blacklist_filtered_item_count",
    "removed_price_outlier_count",
    "removed_price_outlier_rate",
    "clean_item_count",
    "clean_min_price",
    "clean_max_price",
    "clean_average_price",
    "clean_median_price",
]].sort_values("keyword")

clean_keyword_price_summary_df


,keyword,name_blacklist_filtered_item_count,removed_price_outlier_count,removed_price_outlier_rate,clean_item_count,clean_min_price,clean_max_price,clean_average_price,clean_median_price
0,5090,218,45,0.206422,173,1500000,12200000,7.165009e+06,7415000.0
1,a7c2,205,62,0.302439,143,394000,4200000,1.908996e+06,1900000.0
2,ps5,1024,190,0.185547,834,8000,190000,4.357066e+04,33166.5
3,y700,205,38,0.185366,167,70000,1000000,3.964204e+05,370000.0
4,갤럭시,10327,2208,0.213808,8119,32000,710000,2.207634e+05,165000.0
5,게이밍 노트북,923,122,0.132178,801,179000,2980000,1.034314e+06,950000.0
6,골드바,506,185,0.365613,321,165000,1840000,6.984189e+05,820000.0
7,던스,1217,144,0.118324,1073,12000,205000,6.949121e+04,59000.0
8,만년필,1111,300,0.270027,811,30000,1173000,2.742808e+05,160000.0
9,맥미니 m4,100,22,0.220000,78,300000,2000000,1.011513e+06,900000.0


## Keyword 기반 상품명 클러스터링 트리

`clean_price_df`를 입력으로 사용해 크롤링 keyword를 1차 루트로 두고, 각 keyword 내부에서 상품명을 기준으로 클러스터를 만듭니다. 결과는 요약용 DataFrame과 트리 형태 dict 둘 다 확인할 수 있습니다.

In [12]:
from pathlib import Path

from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer


# 클러스터링 조정값입니다. 결과가 너무 잘게 쪼개지면 threshold를 올리고, 너무 뭉치면 낮추세요.
CLUSTER_DISTANCE_THRESHOLD = 0.58
CLUSTER_MIN_ITEMS = 2
CLUSTER_TREE_ITEM_LIMIT = 20
CLUSTER_REVIEW_KEYWORD = None  # 예: "갤럭시", "아이폰". 전체 조회는 None
CLUSTER_REVIEW_LIMIT = 200

# 상품명 매칭에 방해되는 거래/상태/배송 문구입니다.
CLUSTER_TRADE_NOISE_PATTERN = re.compile(
    r"판매합니다|판매해요|판매중|판매완료|판매|"
    r"팝니다|팔아요|팔아봅니다|팔아여|"
    r"구매합니다|구해요|구합니다|삽니다|매입|"
    r"급처합니다|급처|급매|일괄판매|일괄|"
    r"택포|착불|배송|직거래|거래|네고|에눌|"
    r"미개봉|새상품|신품|중고|정품|풀박스|풀박|단품|"
    r"교환|대여|렌탈"
)


def cluster_normalized_name(value):
    text = normalize_search_text(value)
    text = CLUSTER_TRADE_NOISE_PATTERN.sub(" ", text)
    text = re.sub(r"\b\d{1,3}(?:gb|g|tb)\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_keyword_name_clusters(source_df):
    cluster_frames = []
    cluster_tree = []

    for keyword, keyword_df in source_df.groupby("keyword", sort=True):
        keyword_df = keyword_df.copy().reset_index(drop=True)
        keyword_df["cluster_name_text"] = keyword_df["name"].apply(cluster_normalized_name)
        keyword_df.loc[keyword_df["cluster_name_text"] == "", "cluster_name_text"] = keyword_df["name"].astype(str)

        if len(keyword_df) < CLUSTER_MIN_ITEMS:
            keyword_df["cluster_id"] = 0
        else:
            vectorizer = TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(2, 5),
                min_df=1,
                max_features=5000,
            )
            name_vectors = vectorizer.fit_transform(keyword_df["cluster_name_text"])

            try:
                model = AgglomerativeClustering(
                    n_clusters=None,
                    metric="cosine",
                    linkage="average",
                    distance_threshold=CLUSTER_DISTANCE_THRESHOLD,
                )
            except TypeError:
                model = AgglomerativeClustering(
                    n_clusters=None,
                    affinity="cosine",
                    linkage="average",
                    distance_threshold=CLUSTER_DISTANCE_THRESHOLD,
                )

            keyword_df["cluster_id"] = model.fit_predict(name_vectors.toarray())

        keyword_cluster_summaries = []
        keyword_clusters = []
        for cluster_id, cluster_df in keyword_df.groupby("cluster_id", sort=True):
            price_median = cluster_df["price_numeric"].median()
            representative_row = (
                cluster_df.assign(
                    _price_gap=(cluster_df["price_numeric"] - price_median).abs(),
                    _name_length=cluster_df["cluster_name_text"].str.len(),
                )
                .sort_values(["_price_gap", "_name_length", "price_numeric"])
                .iloc[0]
            )
            representative_name = representative_row["cluster_name_text"]
            original_representative_name = representative_row["name"]
            sample_names = list(dict.fromkeys(cluster_df["cluster_name_text"].astype(str).head(5)))
            original_sample_names = list(dict.fromkeys(cluster_df["name"].astype(str).head(5)))

            keyword_cluster_summaries.append(
                {
                    "keyword": keyword,
                    "cluster_id": int(cluster_id),
                    "representative_name": representative_name,
                    "original_representative_name": original_representative_name,
                    "item_count": len(cluster_df),
                    "min_price": int(cluster_df["price_numeric"].min()),
                    "median_price": float(price_median),
                    "average_price": float(cluster_df["price_numeric"].mean()),
                    "max_price": int(cluster_df["price_numeric"].max()),
                    "sample_names": " | ".join(sample_names),
                    "original_sample_names": " | ".join(original_sample_names),
                }
            )
            keyword_clusters.append(
                {
                    "cluster_id": int(cluster_id),
                    "representative_name": representative_name,
                    "original_representative_name": original_representative_name,
                    "item_count": len(cluster_df),
                    "price_summary": {
                        "min": int(cluster_df["price_numeric"].min()),
                        "median": float(price_median),
                        "average": float(cluster_df["price_numeric"].mean()),
                        "max": int(cluster_df["price_numeric"].max()),
                    },
                    "items": cluster_df[
                        ["platform", "pid", "name", "price_numeric", "status", "link"]
                    ].head(CLUSTER_TREE_ITEM_LIMIT).to_dict("records"),
                }
            )

        keyword_cluster_summary_df = pd.DataFrame(keyword_cluster_summaries).sort_values(
            ["keyword", "item_count", "median_price"], ascending=[True, False, True]
        )
        cluster_frames.append(keyword_cluster_summary_df)
        cluster_tree.append(
            {
                "keyword": keyword,
                "item_count": len(keyword_df),
                "cluster_count": len(keyword_cluster_summary_df),
                "clusters": sorted(keyword_clusters, key=lambda item: item["item_count"], reverse=True),
            }
        )

    summary_df = pd.concat(cluster_frames, ignore_index=True) if cluster_frames else pd.DataFrame()
    return summary_df, cluster_tree


keyword_cluster_summary_df, keyword_cluster_tree = build_keyword_name_clusters(clean_price_df)

cluster_review_df = keyword_cluster_summary_df.copy()
if CLUSTER_REVIEW_KEYWORD:
    cluster_review_df = cluster_review_df[
        cluster_review_df["keyword"].astype(str).str.contains(CLUSTER_REVIEW_KEYWORD, case=False, na=False)
    ]

CLUSTER_REVIEW_CSV_PATH = Path("exports") / "cluster_review_df.csv"
CLUSTER_REVIEW_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
cluster_review_df.to_csv(CLUSTER_REVIEW_CSV_PATH, index=False, encoding="utf-8-sig")

print(
    f"keyword 수: {len(keyword_cluster_tree):,}, "
    f"cluster 수: {len(keyword_cluster_summary_df):,}, "
    f"대상 상품 수: {len(clean_price_df):,}, "
    f"CSV 저장: {CLUSTER_REVIEW_CSV_PATH}"
)
cluster_review_df.head(CLUSTER_REVIEW_LIMIT)

keyword 수: 20, cluster 수: 5,107, 대상 상품 수: 19,386, CSV 저장: exports\cluster_review_df.csv


,keyword,cluster_id,representative_name,original_representative_name,item_count,min_price,median_price,average_price,max_price,sample_names,original_sample_names
0,5090,6,7600 a620 rtx5090,7600/a620/RTX5090/1TB,19,7415000,7675000.0,7.970789e+06,9025000,7950x3d a620 rtx5090 | 7600x a620 rtx5090 | 96...,7950X3D/a620/RTX5090/512GB | 7950X3D/a620/RTX5...
1,5090,5,rtx 5090 아스트랄,RTX 5090 아스트랄 팝니다. 미개봉,9,1500000,6400000.0,5.810000e+06,6700000,asus rtx5090 아스트랄 lc | asus rtx 5090 아스트랄 oc |...,Asus RTX5090 아스트랄 LC 미개봉 판매합니다 | ASUS RTX 5090...
2,5090,4,13400 h810 rtx5090,13400/h810/RTX5090/1TB,9,7435000,7745000.0,7.713889e+06,7895000,13400 h810 rtx5090 | 14700 h810 rtx5090 | 1450...,13400/h810/RTX5090/1TB | 14700/h810/RTX5090/51...
3,5090,12,9800x3d 5090 슈프림,9800X3D 5090 슈프림 64GB 2TB,8,8000000,8550000.0,9.031250e+06,10800000,9800x3d 5090 슈프림 | 9800x3d rtx5090 아스트랄 하이엔드 데...,9800X3D 5090 슈프림 64GB 2TB | 9800X3D RTX5090 아스...
4,5090,38,라이젠9 9950x3d 5090 마스터 pc,라이젠9 9950X3D 5090 마스터 PC,8,10000000,10000000.0,1.037500e+07,11000000,라이젠9 9950x3d 5090 y70 최고사양 | 라이젠9 9950x3d rtx5...,라이젠9 9950X3D 5090 Y70 최고사양 | 라이젠9 9950X3D RTX5...
...,...,...,...,...,...,...,...,...,...,...,...
195,ps5,130,ps5 그란투리스모7 정식발매 한글판,(새상품) PS5 그란투리스모7 정식발매 한글판,4,19500,28500.0,4.162500e+04,90000,ps5 그란투리스모7 정식발매 한글판 | ps5 그란투리스모 7 25주년 에디션 |...,(새상품) PS5 그란투리스모7 정식발매 한글판 | ps5 그란투리스모 7 25주년...
196,ps5,155,ps5 카메라,ps5 카메라,4,20000,30000.0,3.000000e+04,40000,ps5 카메라 | ps5 hd 카메라 방송 얼굴캠용 박스포함 | ps5 hd 카메라...,ps5 카메라 | PS5 HD 카메라 정품 (방송/얼굴캠용) 박스포함 | PS5 H...
197,ps5,43,ps5 리애니멀 코드미사용,ps5 리애니멀 코드미사용,4,20000,30800.0,3.215000e+04,47000,ps5 플스5 바하 빌리지 코드미사용 | ps5 레인보우식스 익스트랙션 코드미사용 ...,PS5 플스5 바하 빌리지 코드미사용 판매 | ps5 레인보우식스 익스트랙션(코드미...
198,ps5,31,ps5 프로야구 스피리츠 2024,ps5 프로야구 스피리츠 2024,4,18000,32500.0,2.950000e+04,35000,ps5 풋볼매니저 2024 fm2024 | ps5 프로야구 스피리츠 2024 202...,[미개봉] Ps5 풋볼매니저 2024 fm2024 | PS5 프로야구 스피리츠 20...


## 클러스터링 전 특수문자 리스크 샘플

상품명 정규화 과정에서 의미가 사라질 수 있는 특수문자/기호 패턴을 별도로 확인합니다. 이 결과를 보고 `cluster_normalized_name()`에서 보존하거나 치환할 규칙을 정합니다.

## 최소단계 클러스터링 리뷰

큰 군집으로 묶기 전에, 같은 keyword 안에서 상품명이 거의 같은 항목끼리만 먼저 묶이는 최소단계 결과를 확인합니다. 유사도 기준을 높게 잡아 중복/유사 매물 수준의 작은 묶음을 보는 용도입니다.

In [13]:
# 클러스터링 전 특수문자/기호가 의미를 가질 수 있는 케이스를 별도로 확인합니다.
SPECIAL_CHAR_REVIEW_KEYWORD = None  # 예: "갤럭시", "아이폰". 전체 조회는 None
SPECIAL_CHAR_SAMPLE_LIMIT = 8
SPECIAL_CHAR_REVIEW_LIMIT = 300

SPECIAL_CHAR_PATTERNS = [
    {
        "pattern_id": "plus_symbol_or_word",
        "description": "+/＋/plus: 플러스 모델 구분 가능성",
        "regex": r"(?i)(\+|＋|\bplus\b)",
    },
    {
        "pattern_id": "slash_multiple_models",
        "description": "/: 여러 모델 동시 기재 또는 호환 기종 구분 가능성",
        "regex": r"/",
    },
    {
        "pattern_id": "hyphen_model_token",
        "description": "-: 모델명/세대/등급 연결 가능성",
        "regex": r"(?i)[0-9a-z가-힣]\s*[-–—]\s*[0-9a-z가-힣]",
    },
    {
        "pattern_id": "parentheses_detail",
        "description": "괄호: 상태/구성/세부 모델 정보 포함 가능성",
        "regex": r"[\(\)\[\]\{\}]",
    },
    {
        "pattern_id": "inch_symbol",
        "description": "따옴표/인치: 화면 크기 정보 가능성",
        "regex": r"\d+(?:\.\d+)?\s*(?:\"|”|″|인치)",
    },
    {
        "pattern_id": "storage_unit",
        "description": "GB/TB: 용량 구분 가능성",
        "regex": r"(?i)\d+\s*(?:gb|g|tb)",
    },
    {
        "pattern_id": "roman_generation",
        "description": "I/II/III/IV: 세대 표기 가능성",
        "regex": r"(?i)\b(?:i|ii|iii|iv|v)\b",
    },
    {
        "pattern_id": "ampersand_bundle",
        "description": "&: 번들/동시 기재 가능성",
        "regex": r"&",
    },
    {
        "pattern_id": "remaining_special_chars",
        "description": "기타 특수문자: 정규화 시 공백화되는 잔여 기호",
        "regex": r"[^0-9a-zA-Z가-힣\s]",
    },
]


def matched_special_char_patterns(name):
    text = str(name or "")
    matched = []
    for item in SPECIAL_CHAR_PATTERNS:
        if re.search(item["regex"], text):
            matched.append(item["pattern_id"])
    return matched


special_char_review_base_df = clean_price_df.copy()
if SPECIAL_CHAR_REVIEW_KEYWORD:
    special_char_review_base_df = special_char_review_base_df[
        special_char_review_base_df["keyword"].astype(str).str.contains(
            SPECIAL_CHAR_REVIEW_KEYWORD, case=False, na=False
        )
    ].copy()

special_char_review_df = special_char_review_base_df[
    ["keyword", "platform", "pid", "name", "price_numeric", "status", "link"]
].copy()
special_char_review_df["matched_special_patterns"] = special_char_review_df["name"].apply(
    matched_special_char_patterns
)
special_char_review_df = special_char_review_df[
    special_char_review_df["matched_special_patterns"].apply(bool)
].copy()
special_char_review_df["matched_special_patterns_text"] = special_char_review_df[
    "matched_special_patterns"
].apply(lambda values: " | ".join(values))
special_char_review_df["normalized_for_cluster"] = special_char_review_df["name"].apply(
    cluster_normalized_name if "cluster_normalized_name" in globals() else normalize_search_text
)

special_char_pattern_summary_rows = []
for item in SPECIAL_CHAR_PATTERNS:
    pattern_df = special_char_review_df[
        special_char_review_df["matched_special_patterns"].apply(lambda values: item["pattern_id"] in values)
    ]
    special_char_pattern_summary_rows.append(
        {
            "pattern_id": item["pattern_id"],
            "description": item["description"],
            "matched_row_count": len(pattern_df),
            "keyword_count": pattern_df["keyword"].nunique(),
            "sample_names": " | ".join(
                dict.fromkeys(pattern_df["name"].astype(str).head(SPECIAL_CHAR_SAMPLE_LIMIT))
            ),
        }
    )

special_char_pattern_summary_df = pd.DataFrame(special_char_pattern_summary_rows).sort_values(
    ["matched_row_count", "pattern_id"], ascending=[False, True]
)

print(
    f"특수문자/기호 패턴 포함 상품 수: {len(special_char_review_df):,} / "
    f"검토 대상 상품 수: {len(special_char_review_base_df):,}"
)
display(special_char_pattern_summary_df)

special_char_review_df[
    [
        "keyword",
        "matched_special_patterns_text",
        "name",
        "normalized_for_cluster",
        "price_numeric",
        "platform",
        "status",
        "link",
    ]
].sort_values(["keyword", "matched_special_patterns_text", "price_numeric"]).head(SPECIAL_CHAR_REVIEW_LIMIT)

특수문자/기호 패턴 포함 상품 수: 10,181 / 검토 대상 상품 수: 19,386


,pattern_id,description,matched_row_count,keyword_count,sample_names
8,remaining_special_chars,기타 특수문자: 정규화 시 공백화되는 잔여 기호,9048,20,MSI 게이밍노트북 라이젠7535 RTX4050 상태아주좋음! | [최저가]아수스 ...
3,parentheses_detail,괄호: 상태/구성/세부 모델 정보 포함 가능성,5899,20,[최저가]아수스 18인치 게이밍노트북 TUF(260/5070/1TB) | [최저가]...
5,storage_unit,GB/TB: 용량 구분 가능성,2875,13,ASUS 젠북 S 14 OLED 코어Ultra7 32GB 1TB | [최저가]아수스...
1,slash_multiple_models,/: 여러 모델 동시 기재 또는 호환 기종 구분 가능성,1224,20,[최저가]아수스 18인치 게이밍노트북 TUF(260/5070/1TB) | [최저가]...
2,hyphen_model_token,-: 모델명/세대/등급 연결 가능성,729,17,[최저가]에일리언웨어 게이밍노트북 X17(i7-11/3070/램32/4K | [최저...
0,plus_symbol_or_word,+/＋/plus: 플러스 모델 구분 가능성,728,19,흰둥이노트북+무선마우스+초경량노트북+ 게이밍노트북삼성노트양산 | 삼성 게이밍 노트북...
4,inch_symbol,따옴표/인치: 화면 크기 정보 가능성,308,11,[최저가]아수스 18인치 게이밍노트북 TUF(260/5070/1TB) | ASUS ...
7,ampersand_bundle,&: 번들/동시 기재 가능성,93,15,HP Victus 게이밍 노트북 RTX3050 & 로지텍 G304 마우스 | 미개봉...
6,roman_generation,I/II/III/IV: 세대 표기 가능성,28,8,메탈빌드 I.W.S.P | 반다이 메탈빌드 건담 엑시아 리페어 IV (리페어4) |...


,keyword,matched_special_patterns_text,name,normalized_for_cluster,price_numeric,platform,status,link
5292,5090,hyphen_model_token | parentheses_detail | inch...,"Alienware Area-51(18인치 RTX 5090, 인텔 i9 275H, 1...",alienware area 51 18인치 rtx 5090 인텔 i9 275h 1t ...,7000000,중고나라,판매중,https://web.joongna.com/product/229178283
5276,5090,hyphen_model_token | parentheses_detail | rema...,HP OMEN 45L GT22-3002KL RTX 5090 게이밍 데스크탑(미사용),hp omen 45l gt22 3002kl rtx 5090 게이밍 데스크탑 미사용,9000000,중고나라,판매중,https://web.joongna.com/product/228902400
5218,5090,hyphen_model_token | roman_generation | remain...,i9 14900k Z790-i RTX 5090 FE ITX 미니 데스크탑,i9 14900k z790 i rtx 5090 fe itx 미니 데스크탑,8000000,번개장터,예약중,https://m.bunjang.co.kr/products/403861716
5294,5090,hyphen_model_token | roman_generation | remain...,14900k z790-i DDR5 RTX 5090 FE nzxt 크라켄 ITX 미니...,14900k z790 i ddr5 rtx 5090 fe nzxt 크라켄 itx 미니...,8000000,중고나라,판매중,https://web.joongna.com/product/229073029
5311,5090,parentheses_detail | remaining_special_chars,[구매] 레이저블레이드 16 5090 razerblade 16 구매,구매 레이저블레이드 16 5090 razerblade 16 구매,4800000,중고나라,판매중,https://web.joongna.com/product/225325617
...,...,...,...,...,...,...,...,...
6889,ps5,parentheses_detail | remaining_special_chars,미개봉) PS5 듀얼센스 엣지 모듈,ps5 듀얼센스 엣지 모듈,30000,중고나라,판매중,https://web.joongna.com/product/228546754
6903,ps5,parentheses_detail | remaining_special_chars,(택포3.0)ps5 스텔라 블레이드,3 0 ps5 스텔라 블레이드,30000,중고나라,예약중,https://web.joongna.com/product/229169564
6134,ps5,parentheses_detail | remaining_special_chars,미개봉)최저가)초회특전판)PS5 스칼렛 스트링스,최저가 초회특전판 ps5 스칼렛 스트링스,31000,번개장터,판매중,https://m.bunjang.co.kr/products/333446000
5893,ps5,parentheses_detail | remaining_special_chars,PS5] 진여신전생5 벤전스,ps5 진여신전생5 벤전스,31910,번개장터,판매중,https://m.bunjang.co.kr/products/406603285


In [14]:
from sklearn.neighbors import NearestNeighbors


# 최소단계 묶음 기준입니다. 높일수록 거의 같은 이름만 묶이고, 낮출수록 더 많이 묶입니다.
MIN_STAGE_SIMILARITY_THRESHOLD = 0.92
MIN_STAGE_MIN_ITEMS = 2
MIN_STAGE_TREE_ITEM_LIMIT = 12
MIN_STAGE_REVIEW_KEYWORD = None  # 예: "갤럭시", "아이폰". 전체 조회는 None
MIN_STAGE_REVIEW_LIMIT = 300


def find_connected_components(size, edges):
    parent = list(range(size))

    def find(value):
        while parent[value] != value:
            parent[value] = parent[parent[value]]
            value = parent[value]
        return value

    def union(left, right):
        left_root = find(left)
        right_root = find(right)
        if left_root != right_root:
            parent[right_root] = left_root

    for left, right in edges:
        union(left, right)

    components = {}
    for idx in range(size):
        components.setdefault(find(idx), []).append(idx)
    return list(components.values())


def build_min_stage_name_clusters(source_df):
    summary_rows = []
    tree_rows = []

    for keyword, keyword_df in source_df.groupby("keyword", sort=True):
        keyword_df = keyword_df.copy().reset_index(drop=True)
        keyword_df["cluster_name_text"] = keyword_df["name"].apply(
            cluster_normalized_name if "cluster_normalized_name" in globals() else normalize_search_text
        )
        keyword_df.loc[keyword_df["cluster_name_text"] == "", "cluster_name_text"] = keyword_df[
            "name"
        ].astype(str)

        if len(keyword_df) < MIN_STAGE_MIN_ITEMS:
            continue

        vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(2, 5),
            min_df=1,
            max_features=5000,
        )
        name_vectors = vectorizer.fit_transform(keyword_df["cluster_name_text"])
        radius = 1 - MIN_STAGE_SIMILARITY_THRESHOLD

        neighbor_model = NearestNeighbors(metric="cosine", algorithm="brute", radius=radius)
        neighbor_model.fit(name_vectors)
        distances_list, indices_list = neighbor_model.radius_neighbors(
            name_vectors,
            return_distance=True,
            sort_results=True,
        )

        edges = []
        edge_similarity_by_pair = {}
        for source_idx, (distances, indices) in enumerate(zip(distances_list, indices_list)):
            for distance, target_idx in zip(distances, indices):
                target_idx = int(target_idx)
                if source_idx >= target_idx:
                    continue
                similarity = 1 - float(distance)
                if similarity >= MIN_STAGE_SIMILARITY_THRESHOLD:
                    edges.append((source_idx, target_idx))
                    edge_similarity_by_pair[(source_idx, target_idx)] = similarity

        components = [
            component
            for component in find_connected_components(len(keyword_df), edges)
            if len(component) >= MIN_STAGE_MIN_ITEMS
        ]

        keyword_clusters = []
        for min_stage_cluster_id, component in enumerate(components):
            component_df = keyword_df.iloc[component].copy()
            price_median = component_df["price_numeric"].median()
            representative_row = (
                component_df.assign(
                    _price_gap=(component_df["price_numeric"] - price_median).abs(),
                    _name_length=component_df["cluster_name_text"].str.len(),
                )
                .sort_values(["_price_gap", "_name_length", "price_numeric"])
                .iloc[0]
            )

            component_set = set(component)
            component_similarities = [
                similarity
                for (left, right), similarity in edge_similarity_by_pair.items()
                if left in component_set and right in component_set
            ]
            min_similarity = min(component_similarities) if component_similarities else 1.0
            max_similarity = max(component_similarities) if component_similarities else 1.0
            average_similarity = (
                sum(component_similarities) / len(component_similarities)
                if component_similarities
                else 1.0
            )

            sample_names = list(dict.fromkeys(component_df["name"].astype(str).head(6)))
            normalized_samples = list(
                dict.fromkeys(component_df["cluster_name_text"].astype(str).head(6))
            )
            summary_row = {
                "keyword": keyword,
                "min_stage_cluster_id": int(min_stage_cluster_id),
                "representative_name": representative_row["name"],
                "item_count": len(component_df),
                "min_pair_similarity": round(min_similarity, 4),
                "avg_pair_similarity": round(average_similarity, 4),
                "max_pair_similarity": round(max_similarity, 4),
                "min_price": int(component_df["price_numeric"].min()),
                "median_price": float(price_median),
                "max_price": int(component_df["price_numeric"].max()),
                "sample_names": " | ".join(sample_names),
                "normalized_samples": " | ".join(normalized_samples),
            }
            summary_rows.append(summary_row)
            keyword_clusters.append(
                {
                    **summary_row,
                    "items": component_df[
                        ["platform", "pid", "name", "price_numeric", "status", "link"]
                    ].head(MIN_STAGE_TREE_ITEM_LIMIT).to_dict("records"),
                }
            )

        if keyword_clusters:
            tree_rows.append(
                {
                    "keyword": keyword,
                    "item_count": len(keyword_df),
                    "min_stage_cluster_count": len(keyword_clusters),
                    "clustered_item_count": sum(item["item_count"] for item in keyword_clusters),
                    "clusters": sorted(
                        keyword_clusters,
                        key=lambda item: (item["item_count"], item["avg_pair_similarity"]),
                        reverse=True,
                    ),
                }
            )

    summary_df = pd.DataFrame(summary_rows)
    if not summary_df.empty:
        summary_df = summary_df.sort_values(
            ["keyword", "item_count", "avg_pair_similarity"],
            ascending=[True, False, False],
        ).reset_index(drop=True)
    return summary_df, tree_rows


min_stage_cluster_summary_df, min_stage_cluster_tree = build_min_stage_name_clusters(clean_price_df)

min_stage_review_df = min_stage_cluster_summary_df.copy()
if MIN_STAGE_REVIEW_KEYWORD and not min_stage_review_df.empty:
    min_stage_review_df = min_stage_review_df[
        min_stage_review_df["keyword"].astype(str).str.contains(
            MIN_STAGE_REVIEW_KEYWORD, case=False, na=False
        )
    ]

print(
    f"최소단계 cluster 수: {len(min_stage_cluster_summary_df):,}, "
    f"묶인 상품 수: {min_stage_cluster_summary_df['item_count'].sum() if not min_stage_cluster_summary_df.empty else 0:,}, "
    f"대상 상품 수: {len(clean_price_df):,}, "
    f"유사도 기준: {MIN_STAGE_SIMILARITY_THRESHOLD}"
)
min_stage_review_df.head(MIN_STAGE_REVIEW_LIMIT)

최소단계 cluster 수: 2,303, 묶인 상품 수: 8,494, 대상 상품 수: 19,386, 유사도 기준: 0.92


,keyword,min_stage_cluster_id,representative_name,item_count,min_pair_similarity,avg_pair_similarity,max_pair_similarity,min_price,median_price,max_price,sample_names,normalized_samples
0,5090,25,9950X3D/5090 고성능 게이밍 PC 본체,5,1.0000,1.000,1.0,9000000,9000000.0,9200000,9950X3D/5090 고성능 게이밍 PC 본체,9950x3d 5090 고성능 게이밍 pc 본체
1,5090,15,7600/a620/RTX5090/512GB,4,1.0000,1.000,1.0,7575000,7625000.0,7675000,7600/a620/RTX5090/1TB | 7600/a620/RTX5090/512GB,7600 a620 rtx5090
2,5090,10,14700/h810/RTX5090/512GB,3,1.0000,1.000,1.0,7895000,7895000.0,7895000,14700/h810/RTX5090/512GB | 14700/h810/RTX5090/1TB,14700 h810 rtx5090
3,5090,24,(미개봉) 5090 아스트랄 화이트,3,1.0000,1.000,1.0,6690000,6700000.0,6700000,5090 화이트 아스트랄 | (미개봉) 5090 아스트랄 화이트 | 5090 아스트...,5090 화이트 아스트랄 | 5090 아스트랄 화이트
4,5090,5,"9950x3d, msi 슈프림 5090, 램 64g 올블랙 팝니다",3,0.9594,0.973,1.0,9590000,9590000.0,9590000,"9950x3d, msi 슈프림 5090, 램 64g 올블랙 PC팝니다 | 9950x...",9950x3d msi 슈프림 5090 램 올블랙 pc | 9950x3d msi 슈프...
...,...,...,...,...,...,...,...,...,...,...,...,...
295,갤럭시,579,갤럭시 S22플러스 화이트 무잔상 정상공기기 중고폰,8,1.0000,1.000,1.0,270000,270000.0,270000,갤럭시 S22플러스 화이트 무잔상 정상공기기 중고폰,갤럭시 s22플러스 화이트 무잔상 정상공기기 폰
296,갤럭시,627,갤럭시노트20 5G 256G 그레이(N981) 판매합니다,8,1.0000,1.000,1.0,140000,140000.0,150000,갤럭시노트20 5G 256G 그레이(N981) 판매합니다,갤럭시노트20 그레이 n981
297,갤럭시,664,S급 새상품급 갤럭시Z플립5 최상품 팝니다 정상해지 할인가능,8,1.0000,1.000,1.0,399000,399000.0,399000,S급 새상품급 갤럭시Z플립5 최상품 팝니다 정상해지 할인가능,s급 급 갤럭시z플립5 최상품 정상해지 할인가능
298,갤럭시,722,갤럭시 S21바이올렛 A급 무잔상 정상공기기 중고폰,8,1.0000,1.000,1.0,195000,195000.0,195000,갤럭시 S21바이올렛 A급 무잔상 정상공기기 중고폰,갤럭시 s21바이올렛 a급 무잔상 정상공기기 폰
